In [11]:
!pip install pandas numpy scikit-learn

In [13]:
!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

--2026-04-29 08:58:24--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  21.5MB/s    in 0.2s    

2026-04-29 08:58:25 (21.5 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

unzip:  cannot find or open ml-100k.zip!wget, ml-100k.zip!wget.zip or ml-100k.zip!wget.ZIP.
Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflatin

In [15]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [17]:
ratings = pd.read_csv(
    'ml-100k/u.data',
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

print("Ratings Data:")
print(ratings.head())

Ratings Data:
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596


In [19]:
movies = pd.read_csv(
    'ml-100k/u.item',
    sep='|',
    encoding='latin-1',
    header=None,
    usecols=[0, 1],
    names=['item_id', 'title']
)

print("\nMovies Data:")
print(movies.head())


Movies Data:
   item_id              title
0        1   Toy Story (1995)
1        2   GoldenEye (1995)
2        3  Four Rooms (1995)
3        4  Get Shorty (1995)
4        5     Copycat (1995)


In [21]:
data = pd.merge(ratings, movies, on='item_id')

print("\nMerged Data:")
print(data.head())


Merged Data:
   user_id  item_id  rating  timestamp                       title
0      196      242       3  881250949                Kolya (1996)
1      186      302       3  891717742    L.A. Confidential (1997)
2       22      377       1  878887116         Heavyweights (1994)
3      244       51       2  880606923  Legends of the Fall (1994)
4      166      346       1  886397596         Jackie Brown (1997)


In [23]:
data_small = data[data['user_id'] <= 500]  # smaller = faster
print("\nFiltered Data Shape:", data_small.shape)


Filtered Data Shape: (56770, 5)


In [25]:
movie_matrix = data_small.pivot_table(
    index='title',
    columns='user_id',
    values='rating'
).fillna(0)

print("\nMatrix Shape:", movie_matrix.shape)


Matrix Shape: (1604, 500)


In [27]:
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(movie_matrix)

print("\nModel Trained Successfully!")


Model Trained Successfully!


In [28]:
movie_name = "Star Wars (1977)"

In [29]:
if movie_name in movie_matrix.index:

    query_index = movie_matrix.index.get_loc(movie_name)

    distances, indices = model_knn.kneighbors(
        movie_matrix.iloc[query_index, :].values.reshape(1, -1),
        n_neighbors=6
    )

    print(f"\n🎬 Recommendations for '{movie_name}':\n")

    for i in range(1, len(distances.flatten())):
        print(f"{i}. {movie_matrix.index[indices.flatten()[i]]} "
              f"(Similarity: {1 - distances.flatten()[i]:.2f})")

else:
    print("Movie not found in dataset!")


🎬 Recommendations for 'Star Wars (1977)':

1. Return of the Jedi (1983) (Similarity: 0.88)
2. Raiders of the Lost Ark (1981) (Similarity: 0.79)
3. Empire Strikes Back, The (1980) (Similarity: 0.76)
4. Toy Story (1995) (Similarity: 0.73)
5. Fargo (1996) (Similarity: 0.73)
